Code below read file from gcs and make some sql above it 

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [2]:
credentials_location = '/workspaces/Data-Engineering/creds.json'

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", "./lib/gcs-connector-hadoop3-2.2.5.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

In [3]:
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 21:37:46 WARN Utils: Your hostname, codespaces-886ad4, resolves to a loopback address: 127.0.0.1; using 10.0.0.220 instead (on interface eth0)
26/04/22 21:37:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/04/22 21:37:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [21]:
df_green = spark.read.parquet('gs://spark-2025-pq')

In [6]:
df_green.count()

4181444

In [7]:
df_green.repartition(4).write.parquet('/tmp/homework')

In [8]:
df_green.registerTempTable('trips')


/workspaces/Data-Engineering/.venv/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [19]:
df_result = spark.sql("""
    SELECT MAX(
        (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600   
    )
    FROM trips  
""").show()

+--------------------------------------------------------------------------------------------------------------------------------------+
|max(((unix_timestamp(tpep_dropoff_datetime, yyyy-MM-dd HH:mm:ss) - unix_timestamp(tpep_pickup_datetime, yyyy-MM-dd HH:mm:ss)) / 3600))|
+--------------------------------------------------------------------------------------------------------------------------------------+
|                                                                                                                     90.64666666666666|
+--------------------------------------------------------------------------------------------------------------------------------------+



In [12]:
df_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [34]:
# wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
from pyspark.sql import types
schema = types.StructType([
        types.StructField("LocationID", types.IntegerType(), True),
        types.StructField("Borough", types.StringType(), True),
        types.StructField("Zone", types.StringType(), True),
        types.StructField("service_zone", types.StringType(), True)
])

df_zones = spark.read \
            .option("header", "true") \
            .schema(schema) \
            .csv('/tmp/taxi_zone_lookup.csv')

In [36]:
df_joined = df_zones.join(df_green, df_green.PULocationID == df_zones.LocationID)

In [37]:
df_joined.createOrReplaceTempView("trips_zones")

In [43]:
df_result = spark.sql("""
    SELECT Zone, COUNT(*)
    FROM trips_zones  
    GROUP BY PULocationID, Zone
    ORDER BY COUNT(*) ASC
    LIMIT 5
""").show()

+--------------------+--------+
|                Zone|count(1)|
+--------------------+--------+
|       Arden Heights|       1|
|Eltingville/Annad...|       1|
|Governor's Island...|       1|
|       Port Richmond|       3|
| Green-Wood Cemetery|       4|
+--------------------+--------+



In [44]:
spark.stop()